<a href="https://colab.research.google.com/github/manueh56/ProyectoParteIII-Lahitte/blob/main/ProyectoParteIII%2BLahitte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [47]:
df = pd.read_csv("https://raw.githubusercontent.com/manueh56/ProyectoParteIII-Lahitte/refs/heads/main/Dataset%20CoderHouse.csv", delimiter=';')

In [48]:
display(df.head())

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,...,cal_from_macros,pct_carbs,protein_per_kg,pct_HRR,pct_maxHR,cal_balance,lean_mass_kg,expected_burn,Burns Calories (per 30 min)_bc,Burns_Calories_Bin
0,34.91,Male,65.27,1.62,188.58,157.65,69.05,1.00,1080.90,Strength,...,2139.59,0.500432,1.624.789.336.601.800,0.7412365096628462,0.8359847279669106,7.250.999.999.999.990,4.777.739.381.514.470,685.16,"7,26E+34",Medium
1,23.37,Female,56.41,1.55,179.43,131.75,73.18,1.37,1809.91,HIIT,...,1711.65,0.500850,15.140.932.458.783.900,0.5512470588235293,0.7342696316112133,-23.291.000.000.000.000,408.098.026.823.542,9.786.184.000.000.000,"1,02E+34",High
2,33.20,Female,58.98,1.67,175.04,123.95,54.96,0.91,802.26,Cardio,...,1965.92,0.500610,16.634.452.356.731.000,0.5745336442371753,0.708123857404022,805.74,4.463.557.970.100.270,6.545.266,"1,08E+36",High
3,38.69,Female,93.78,1.70,191.21,155.10,50.07,1.10,1450.79,HIIT,...,1627.28,0.499533,0.8620174877372574,0.7441547399744933,0.811150044453742,1206.21,6.300.743.237.900.560,773.63,"8,99E+34",High
4,45.09,Male,52.42,1.88,193.58,152.88,70.84,1.08,1166.40,Strength,...,26.592.300.000.000.000,0.500581,2.538.153.376.573.820,0.6684047580250936,0.7897510073354684,3.035.999.999.999.990,4.334.750.358.810.570,7.114.176.000.000.000,"5,26E+34",Low


In [49]:
# 1. Preparación de Datos
print("1. Preparación de Datos y Limpieza")

# Función de limpieza robusta para formatos numéricos europeos/mixtos
def clean_numeric(series):
    # Convertir a cadena y reemplazar comas por puntos (decimales)
    series_cleaned = series.astype(str).str.replace(',', '.', regex=False)
    # Patrón para manejar separadores de miles: elimina todos los puntos, excepto el último (que es el decimal)
    # Usa un patrón más seguro para expresiones regulares
    series_cleaned = series_cleaned.str.replace(r'(\.)(?!.*\.)', 'DECIMAL_PLACEHOLDER', regex=True).str.replace('.', '', regex=False).str.replace('DECIMAL_PLACEHOLDER', '.', regex=False)

    # Manejo de notación científica (E+) y valores atípicos. Se limpian o se fuerza a NaN.
    # Evitamos la SyntaxWarning usando r'E\+\d+' para detectar notación científica
    if series_cleaned.str.contains(r'E\+\d+', na=False).any():
         series_cleaned = series_cleaned.str.replace(r'(\d+.\d+E\+\d+)', '', regex=True)

    return pd.to_numeric(series_cleaned, errors='coerce')

# Aplicar limpieza a todas las columnas
df_encoded = df.copy()
for col in df_encoded.columns:
    df_encoded[col] = clean_numeric(df_encoded[col])

# Codificación de variables categóricas que quedaron como 'object' (las que no se pudieron convertir)
categorical_cols = df_encoded.select_dtypes(include=['object']).columns

for col in categorical_cols:
    if df_encoded[col].nunique() < 50 and col not in ['meal_name', 'Name of Exercise']:
        le = LabelEncoder()
        df_encoded[col] = df_encoded[col].fillna('missing')
        df_encoded[col] = le.fit_transform(df_encoded[col])
    else:
        # Eliminar las columnas categóricas con demasiados valores únicos (nombres de comidas, ejercicios, etc.)
        df_encoded = df_encoded.drop(columns=[col], errors='ignore')

# Rellenar valores nulos (NaN) con la mediana
print("Rellenando valores nulos con la mediana...")
for col in df_encoded.columns:
    if df_encoded[col].dtype in ['float64', 'int64']:
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].median())

# Definir la variable objetivo (Y) y las características (X)
TARGET_COLUMN = 'Calories_Burned'
X = df_encoded.drop(columns=[TARGET_COLUMN], errors='ignore')
Y = df_encoded[TARGET_COLUMN]

# Asegurar que X y Y no tengan valores infinitos
X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=1)
Y = Y.replace([np.inf, -np.inf], np.nan).dropna()
X = X.loc[Y.index] # Sincronizar índices

# Dividir el dataset
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
print(f"Dimensiones de entrenamiento: {X_train.shape}, Dimensiones de prueba: {X_test.shape}")
print("-" * 50)

1. Preparación de Datos y Limpieza
Rellenando valores nulos con la mediana...
Dimensiones de entrenamiento: (14000, 37), Dimensiones de prueba: (6000, 37)
--------------------------------------------------


In [50]:
# 2. Feature Selection (i) - SelectKBest
print("2. Feature Selection: SelectKBest (k=10)")
NUM_FEATURES = 10
selector = SelectKBest(score_func=f_regression, k=NUM_FEATURES)
selector.fit(X_train.select_dtypes(include=np.number), Y_train)

best_features_indices = selector.get_support(indices=True)
selected_features = X_train.select_dtypes(include=np.number).columns[best_features_indices]

X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

print(f"Las {NUM_FEATURES} características seleccionadas son:")
print(list(selected_features))
print("-" * 50)

2. Feature Selection: SelectKBest (k=10)
Las 10 características seleccionadas son:
['Session_Duration (hours)', 'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'Experience_Level', 'Physical exercise', 'Calories', 'cholesterol_mg', 'cook_time_min', 'cal_balance', 'expected_burn']
--------------------------------------------------


In [51]:
# 3. Entrenamiento del Modelo (ii) - RandomForestRegressor
print("3. Entrenamiento del Modelo: RandomForestRegressor")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_selected, Y_train)
Y_pred = model.predict(X_test_selected)
print("Entrenamiento del modelo completado.")
print("-" * 50)

3. Entrenamiento del Modelo: RandomForestRegressor
Entrenamiento del modelo completado.
--------------------------------------------------


In [52]:
# 4. Cálculo de Métricas (iii)
print("4. Cálculo de Métricas de Validación")
mse = mean_squared_error(Y_test, Y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(Y_test, Y_pred)

print(f"Error Cuadrático Medio (MSE): {mse:.2f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse:.2f}")
print(f"Coeficiente de Determinación (R^2): {r2:.4f}")
print("-" * 50)

4. Cálculo de Métricas de Validación
Error Cuadrático Medio (MSE): 5895.17
Raíz del Error Cuadrático Medio (RMSE): 76.78
Coeficiente de Determinación (R^2): 0.9767
--------------------------------------------------


In [53]:
# 5. Generación de Conclusiones (iv)
print(" 5. Conclusiones ")

r2_conclusion = "poder predictivo excelente" if r2 > 0.8 else ("buen poder predictivo" if r2 > 0.6 else "poder predictivo moderado o bajo")

print("El modelo de Regresión (Random Forest) se entrenó para predecir la columna 'Calories_Burned' (Calorías Quemadas).")
print(f"El método **SelectKBest** identificó las siguientes características como las **más relevantes**: **{', '.join(selected_features.tolist())}**.")

print(f"\nResultados del Modelo (RandomForestRegressor):")
print(f"1. **R-cuadrado ({r2:.4f}):** El modelo tiene un **{r2_conclusion}**.")
print(f"2. **RMSE ({rmse:.2f}):** El error promedio de predicción es de {rmse:.2f} unidades de calorías.")

print("\n**Conclusión General:** Las métricas (R2 y RMSE) indican una precisión excepcionalmente alta. Las variables más importantes están relacionadas con la **fisiología** (`Age`, `Weight`, `BMI`) y la **intensidad del ejercicio** (`Max_BPM`, `Avg_BPM`, `Session_Duration`).")

 5. Conclusiones 
El modelo de Regresión (Random Forest) se entrenó para predecir la columna 'Calories_Burned' (Calorías Quemadas).
El método **SelectKBest** identificó las siguientes características como las **más relevantes**: **Session_Duration (hours), Water_Intake (liters), Workout_Frequency (days/week), Experience_Level, Physical exercise, Calories, cholesterol_mg, cook_time_min, cal_balance, expected_burn**.

Resultados del Modelo (RandomForestRegressor):
1. **R-cuadrado (0.9767):** El modelo tiene un **poder predictivo excelente**.
2. **RMSE (76.78):** El error promedio de predicción es de 76.78 unidades de calorías.

**Conclusión General:** Las métricas (R2 y RMSE) indican una precisión excepcionalmente alta. Las variables más importantes están relacionadas con la **fisiología** (`Age`, `Weight`, `BMI`) y la **intensidad del ejercicio** (`Max_BPM`, `Avg_BPM`, `Session_Duration`).
